# Boston Housing Price Prediction with Apache Spark

このNotebookでは、Apache Sparkを使用してBostonデータセット（住宅価格データ）を探索し、高度な回帰モデルを構築します。

特徴量エンジニアリングとデータ標準化を活用した実践的なモデル構築を学びます。

In [ ]:
// Spark 依存関係の読み込み（Scala 2.13 を明示的に指定）
import $ivy.`org.apache.spark:spark-sql_2.13:3.5.0`
import $ivy.`org.apache.spark:spark-mllib_2.13:3.5.0`

println("Spark 依存関係が正常にロードされました")

## 1. 環境設定とライブラリのインポート

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.regression.LinearRegression
import org.apache.spark.ml.feature.{SQLTransformer, VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder}
import org.apache.spark.ml.evaluation.RegressionEvaluator

// SparkSessionの作成
val spark = SparkSession.builder()
  .appName("BostonExploration")
  .master("local[*]")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

println("Spark Session created successfully!")
println(s"Spark version: ${spark.version}")

## 2. データの読み込み

In [ ]:
// データの読み込み
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("../data/Boston.csv")

println(s"データ件数: ${df.count()}")
println("\nスキーマ:")
df.printSchema()

## 3. データの概要確認

In [ ]:
// 最初の10行を表示
df.show(10, truncate = false)

In [ ]:
// 統計情報
df.describe("RM", "LSTAT", "PTRATIO", "PRICE").show()

In [ ]:
// CRIME カテゴリの確認
df.groupBy("CRIME").count().orderBy("count").show()

## 4. カテゴリカル変数のエンコーディング

In [ ]:
// CRIME カテゴリカル変数のエンコーディング
val crimeIndexer = new StringIndexer()
  .setInputCol("CRIME")
  .setOutputCol("CRIME_index")

val crimeEncoder = new OneHotEncoder()
  .setInputCol("CRIME_index")
  .setOutputCol("CRIME_vec")

val encodePipeline = new Pipeline().setStages(Array(
  crimeIndexer, crimeEncoder
))

val encodedDF = encodePipeline.fit(df).transform(df)

println("カテゴリカル変数のエンコーディング完了")
encodedDF.select("CRIME", "CRIME_index", "CRIME_vec").show(5, truncate = false)

## 5. 特徴量エンジニアリング

SQLTransformerを使用して、非線形な関係や交互作用を捉える特徴量を作成します。

In [ ]:
// 特徴量エンジニアリング
val featureEngineering = new SQLTransformer().setStatement("""
  SELECT *,
    RM * RM as RM2,
    LSTAT * LSTAT as LSTAT2,
    PTRATIO * PTRATIO as PTRATIO2,
    RM * LSTAT as RM_LSTAT,
    RM * PTRATIO as RM_PTRATIO
  FROM __THIS__
""")

val engineeredDF = featureEngineering.transform(encodedDF)

println("特徴量エンジニアリング完了")
engineeredDF.select("RM", "RM2", "LSTAT", "LSTAT2", "RM_LSTAT").show(5)

## 6. データクリーニング

In [ ]:
// 外れ値除去
val cleanedDF = engineeredDF
  .filter("PRICE < 50 AND PRICE > 0")
  .filter("RM > 0 AND LSTAT > 0")

println(s"データクリーニング:")
println(s"  元のデータ: ${engineeredDF.count()} 件")
println(s"  クリーニング後: ${cleanedDF.count()} 件")
println(s"  除去: ${engineeredDF.count() - cleanedDF.count()} 件")

## 7. 特徴量の統合

In [ ]:
// 全ての特徴量を1つのベクトルに統合
val assembler = new VectorAssembler()
  .setInputCols(Array(
    // カテゴリカル特徴量（エンコード済み）
    "CRIME_vec",
    // 数値特徴量
    "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS",
    "RAD", "TAX", "PTRATIO", "B", "LSTAT",
    // エンジニアリングした特徴量
    "RM2", "LSTAT2", "PTRATIO2", "RM_LSTAT", "RM_PTRATIO"
  ))
  .setOutputCol("features")
  .setHandleInvalid("skip")

val assembledDF = assembler.transform(cleanedDF)

println(s"準備後のデータ件数: ${assembledDF.count()}")
assembledDF.select("features", "PRICE").show(5, truncate = false)

## 8. データの標準化

StandardScalerを使用して、スケールの異なる特徴量を標準化します。

In [ ]:
// StandardScalerによるデータ標準化
val scaler = new StandardScaler()
  .setInputCol("features")
  .setOutputCol("scaled_features")
  .setWithMean(true)   // 平均を0にする
  .setWithStd(true)    // 標準偏差を1にする

val scaledDF = scaler.fit(assembledDF).transform(assembledDF)

println("データ標準化完了")
scaledDF.select("features", "scaled_features").show(3, truncate = false)

## 9. データの分割

In [ ]:
// 訓練データとテストデータに分割
val Array(trainData, testData) = scaledDF.randomSplit(Array(0.7, 0.3), seed = 42)

println(s"訓練データ: ${trainData.count()} 件")
println(s"テストデータ: ${testData.count()} 件")

## 10. Linear Regressionモデルの訓練

In [ ]:
// Linear Regressionモデルの作成
val lr = new LinearRegression()
  .setLabelCol("PRICE")
  .setFeaturesCol("scaled_features")
  .setMaxIter(100)
  .setRegParam(0.1)        // L2正則化
  .setElasticNetParam(0.0)  // 0=Ridge, 1=Lasso

val pipeline = new Pipeline().setStages(Array(lr))

// モデルの訓練
println("モデルを訓練中...")
val model = pipeline.fit(trainData)
println("訓練完了！")

## 11. モデルの評価

In [ ]:
// テストデータで予測
val predictions = model.transform(testData)

// 評価メトリクスの計算
val evaluator = new RegressionEvaluator()
  .setLabelCol("PRICE")
  .setPredictionCol("prediction")

val r2 = evaluator.setMetricName("r2").evaluate(predictions)
val rmse = evaluator.setMetricName("rmse").evaluate(predictions)
val mae = evaluator.setMetricName("mae").evaluate(predictions)

println(f"R² Score: ${r2 * 100}%.2f%%")
println(f"RMSE: $rmse%.4f")
println(f"MAE: $mae%.4f")

## 12. 予測結果の確認

In [ ]:
// 予測結果のサンプル表示
predictions.select(
  "CRIME", "RM", "LSTAT", "PTRATIO",
  "PRICE", "prediction"
).show(15, truncate = false)

In [ ]:
// 実際の値と予測値の比較（誤差を計算）
import org.apache.spark.sql.functions._

val comparison = predictions.select(
  col("PRICE").as("actual"),
  col("prediction"),
  abs(col("PRICE") - col("prediction")).as("error"),
  (abs(col("PRICE") - col("prediction")) / col("PRICE") * 100).as("error_pct")
)

comparison.describe("actual", "prediction", "error", "error_pct").show()

## 13. モデルの係数確認

In [ ]:
// Linear Regressionモデルの係数を表示
val lrModel = model.stages(0).asInstanceOf[org.apache.spark.ml.regression.LinearRegressionModel]

println("モデルの係数:")
println(s"Intercept: ${lrModel.intercept}")
println(s"Coefficients: ${lrModel.coefficients}")
println()
println("訓練セットでの性能:")
println(s"RMSE: ${lrModel.summary.rootMeanSquaredError}")
println(s"R²: ${lrModel.summary.r2}")

## 14. 特徴量の重要度確認

In [ ]:
// 係数の絶対値を特徴量の重要度として表示
val featureNames = Array("CRIME_vec") ++ 
                   Array("ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT") ++
                   Array("RM2", "LSTAT2", "PTRATIO2", "RM_LSTAT", "RM_PTRATIO")

val coefficients = lrModel.coefficients.toArray

println("特徴量の重要度（係数の絶対値）:")
featureNames.zip(coefficients).sortBy(-_._2.abs).take(10).foreach { case (name, coef) =>
  println(f"$name%-20s: $coef%.4f")
}

## 15. クリーンアップ

In [ ]:
// SparkSessionの停止
// spark.stop()
println("完了！")